# SFace: Matriks Galeri 100 x 128 (Per Foto Individual) & Pencocokan Vektor

Alih-alih hanya 1 vektor rata-rata, galeri disimpan sebagai **matriks $100 \times 128$** per orang:
- **Matriks Faza** : Ukuran $(100, 128)$ $\rightarrow$ representasi 100 foto individual.
- **Matriks Raffin**: Ukuran $(100, 128)$ $\rightarrow$ representasi 100 foto individual.
- **Export CSV**: 200 baris $\times$ 128 kolom disimpan ke `embeddings_100x128.csv`.

### Keunggulan Matriks $(100, 128)$ saat Pencocokan (Inference):
Hanya dengan **1 kali perkalian matriks** (dot product):
$$\text{scores} = M_{100 \times 128} \cdot v_Q$$
Kita langsung mendapatkan **100 nilai Cosine Similarity** sekaligus (kemiripan terhadap setiap pose/foto di galeri) dalam hitungan milidetik!

In [ ]:
# 1. Setup Model Detektor YuNet & Recognizer SFace
from pathlib import Path
import csv
import cv2
import numpy as np

detector = cv2.FaceDetectorYN.create("face_detection_yunet_2023mar.onnx", "", (320, 320), score_threshold=0.6)
recognizer = cv2.FaceRecognizerSF.create("models/face_recognition_sface_2021dec.onnx", "")

def extract_embedding(img_path_or_array, max_dim=1024):
    img = cv2.imread(str(img_path_or_array)) if isinstance(img_path_or_array, (str, Path)) else img_path_or_array
    if img is None:
        return None
    h, w = img.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
        h, w = img.shape[:2]
    detector.setInputSize((w, h))
    _, faces = detector.detect(img)
    if faces is None or len(faces) == 0:
        return None
    aligned = recognizer.alignCrop(img, faces[0])
    feat = recognizer.feature(aligned)[0]
    return feat / np.linalg.norm(feat)

print("✓ Model YuNet & SFace siap.")

In [ ]:
# 2. Ekstraksi Setiap Foto Menjadi Matriks (100, 128)
def extract_person_matrix(folder_path, label):
    p = Path(folder_path)
    files = sorted([f for f in p.glob("*.*") if f.suffix.lower() in [".jpg", ".png", ".jpeg"]])
    
    embeddings_list = []
    csv_rows = []
    
    for f in files:
        emb = extract_embedding(f)
        if emb is not None:
            embeddings_list.append(emb)
            csv_rows.append([label, f.name] + emb.tolist())
            
    # Matriks ukuran (N_foto, 128)
    matrix_100x128 = np.array(embeddings_list, dtype=np.float32)
    return matrix_100x128, csv_rows

print("Mengekstrak 100 foto Faza...")
mat_faza, rows_faza = extract_person_matrix("Faza", "Faza")

print("Mengekstrak 100 foto Raffin...")
mat_raffin, rows_raffin = extract_person_matrix("Raffin", "Raffin")

print(f"✓ Matriks Galeri Faza   : Shape = {mat_faza.shape} (100 baris x 128 kolom)")
print(f"✓ Matriks Galeri Raffin : Shape = {mat_raffin.shape} (100 baris x 128 kolom)")

In [ ]:
# 3. Simpan Seluruh 200 Baris x 128 Dimensi ke CSV
csv_filename = "embeddings_100x128.csv"
header = ["identitas", "file_foto"] + [f"dim_{i}" for i in range(128)]

try:
    with open(csv_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows_faza)
        writer.writerows(rows_raffin)
    print(f"✓ Berhasil menyimpan {len(rows_faza) + len(rows_raffin)} baris foto ke '{csv_filename}'.")
except PermissionError:
    alt_file = "embeddings_100x128_alt.csv"
    with open(alt_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows_faza)
        writer.writerows(rows_raffin)
    print(f"✓ '{csv_filename}' sedang dibuka Excel. Tersimpan ke '{alt_file}'.")

In [ ]:
# 4. Capture Wajah Uji Baru Langsung dari Webcam
cap = cv2.VideoCapture(0)
v_query_live = None

if cap.isOpened():
    print("Tekan [SPASI] di jendela preview untuk capture wajah uji, [ESC] untuk batal.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.putText(frame, "Tekan [SPASI] Capture Live | [ESC] Batal", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imshow("Live Capture Test Face", frame)
        k = cv2.waitKey(1) & 0xFF
        if k == 32:  # SPASI
            v_query_live = extract_embedding(frame)
            if v_query_live is not None:
                print("✓ Wajah berhasil di-capture langsung dari kamera!")
                break
        elif k == 27:  # ESC
            break
    cap.release()
    cv2.destroyAllWindows()

# Fallback simulasi jika kamera non-GUI
if v_query_live is None:
    v_query_live = extract_embedding("../faza/IMG_20261026_213805_247.jpg")

print(f"Vektor Uji Live (v_Q) : Shape={v_query_live.shape}, L2-Norm={np.linalg.norm(v_query_live):.4f}")

In [ ]:
# 5. Pencocokan Matriks: 1 Operasi Perkalian Menghasilkan 100 Skor Kemiripan
# scores = Matrix (100, 128) dot Vector (128,) -> Array (100,)
scores_faza = np.dot(mat_faza, v_query_live)      # 100 skor kemiripan ke semua foto Faza
scores_raffin = np.dot(mat_raffin, v_query_live)  # 100 skor kemiripan ke semua foto Raffin

def analyze_scores(scores, label_name, threshold=0.363):
    best_sim = float(np.max(scores))
    best_idx = int(np.argmax(scores))
    top5_sim = float(np.mean(np.sort(scores)[-5:]))
    mean_sim = float(np.mean(scores))
    
    print(f"\n{'=' * 65}")
    print(f" HASIL KOMPARASI TERHADAP 100 FOTO: {label_name}")
    print(f"{'=' * 65}")
    print(f"  • Kemiripan Tertinggi (Max Score) : {best_sim:.4f}  (Foto ke-{best_idx})")
    print(f"  • Rata-rata 5 Foto Terdekat (Top-5): {top5_sim:.4f}")
    print(f"  • Rata-rata Seluruh 100 Foto       : {mean_sim:.4f}")
    print(f"  • Status Identitas (>= {threshold})    : {'[IDENTIK / COCOK]' if best_sim >= threshold else '[TOLAK / BEDA ORANG]'}")

analyze_scores(scores_faza, "Galeri Faza (100 Foto)")
analyze_scores(scores_raffin, "Galeri Raffin (100 Foto)")